# Fire2Air Darwin — Assessment 3
## 05 Visualisation, Integration, Tomorrow Smoke Outlook and Documentation
**Member:** Dieu Yen Diep

Assessment 3 upgrades:
- visualises FSI, aligned/upwind FRP and next-day PM2.5 relationships;
- consolidates model, station, cross-station and ablation results;
- integrates the three predictions into a station-specific Tomorrow Smoke Outlook;
- creates dashboard-ready decision-support output with explicit reliability rules;
- updates the evidence manifest and README for the Assessment 3 run order;
- keeps API/dashboard deployment as optional future layers unless actually implemented.

In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
%pip install boto3

   ---------------------------------------- 0.0/16.0 MB ? eta -:--:--
   ------------- -------------------------- 5.5/16.0 MB 29.8 MB/s eta 0:00:01
   ---------------------------------------- 16.0/16.0 MB 40.2 MB/s eta 0:00:00

  Attempting uninstall: botocore

    Found existing installation: botocore 1.36.3

   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
    Uninstalling botocore-1.36.3:
   ---------------------------------------- 0/3 [botocore]
      Successfully uninstalled botocore-1.36.3
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.19.0 requires botocore<1.36.4,>=1.36.0, but you have botocore 1.43.108 which is incompatible.


In [5]:
# ============================================================
# AWS S3 CONFIGURATION
# ============================================================
import boto3
import shutil

BUCKET = "fire2air-darwin-dan5-646468992009-ap-southeast-2-an"

NTEPA_RAW = f"s3://{BUCKET}/raw/ntepa/"
FIRMS_RAW = f"s3://{BUCKET}/raw/firms/"
NTEPA_DAILY = f"s3://{BUCKET}/processed/ntepa_daily/"
FIRMS_DAILY = f"s3://{BUCKET}/processed/firms_daily/"
MODEL_READY = f"s3://{BUCKET}/processed/model_ready/"
MODEL1_DIR = f"s3://{BUCKET}/models/model1/"
MODEL2_DIR = f"s3://{BUCKET}/models/model2/"
MODEL3_DIR = f"s3://{BUCKET}/models/model3/"
MODEL1_OUTPUT = f"s3://{BUCKET}/outputs/model1/"
MODEL2_OUTPUT = f"s3://{BUCKET}/outputs/model2/"
MODEL3_OUTPUT = f"s3://{BUCKET}/outputs/model3/"
VIS_OUTPUT = f"s3://{BUCKET}/outputs/visualisation/"
DASHBOARD_DIR = f"s3://{BUCKET}/dashboard/"
DOCUMENTATION_DIR = f"s3://{BUCKET}/documentation/"
ATHENA_RESULTS = f"s3://{BUCKET}/athena-results/"

s3 = boto3.client("s3")
LOCAL_ROOT = Path("/tmp/fire2air_a3")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _prefix(uri):
    return uri.replace(f"s3://{BUCKET}/", "", 1)

def download_prefix(uri, local_dir, flatten=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    downloaded = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel = Path(key[len(prefix):].lstrip("/"))
            dest = local_dir / (rel.name if flatten else rel)
            dest.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(BUCKET, key, str(dest))
            downloaded.append(dest)
    return downloaded

def upload_file(local_path, uri, object_name=None):
    local_path = Path(local_path)
    prefix = _prefix(uri).rstrip("/")
    key = f"{prefix}/{object_name or local_path.name}"
    s3.upload_file(str(local_path), BUCKET, key)
    print(f"Uploaded: s3://{BUCKET}/{key}")

def upload_tree(local_dir, uri):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return
    prefix = _prefix(uri).rstrip("/")
    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(p), BUCKET, key)
    print(f"Uploaded folder: {local_dir} -> {uri}")